# Episode 20: Deploy to Production (GCP)

Your agent works on your laptop. Now let's make it work for everyone else.

**By the end of this episode, you will be able to:**
- Containerize an AG2 agent with Docker and FastAPI
- Deploy to Google Cloud Run
- Apply a production checklist to any agent deployment

## From Development to Production

Throughout this workshop, you've been running agents locally. Production deployment changes the game. Gartner predicts that by 2026, 40% of enterprise applications will embed agentic AI — which means the ability to deploy agents reliably is becoming a core engineering skill, not a nice-to-have.

Here's what shifts when you move from your machine to the cloud.

| Concern | Dev | Production |
|---------|-----|------------|
| **Configuration** | `.env` file | Environment variables / secret manager |
| **Secrets** | Local `.env` | GCP Secret Manager, AWS Secrets Manager |
| **Health** | Manual testing | Health check endpoints, uptime monitoring |
| **Shutdown** | Ctrl+C | Graceful shutdown with in-flight request handling |
| **Scaling** | Single process | Auto-scaling, load balancing |

You have three main deployment options, and the right choice depends on your scale and complexity.

- **Cloud Run (serverless)** — Best for most agent workloads. Scales to zero, pay per request. This is where you should start.
- **Kubernetes (GKE)** — For complex orchestration needs: multiple services, custom networking, persistent connections.
- **VMs (Compute Engine)** — Full control, but you manage everything. Rarely the right choice for agents.

## Part 1 — Containerize Your Agent

If you've deployed a Flask app, this will feel familiar. Same pattern, different payload. Instead of serving web pages, you're serving agent responses behind an HTTP API.

### The Dockerfile

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .
EXPOSE 8080
CMD ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8080"]
```

Three things worth noting: `python:3.12-slim` gives you a smaller image and faster deploys, `EXPOSE 8080` matches Cloud Run's default expectation, and `uvicorn` is a production-grade ASGI server that handles concurrency well.

### The Server: FastAPI + AG2

Wrap your agent in a FastAPI application. This gives you HTTP endpoints, automatic OpenAPI docs, and async support — everything you need for a production API.

```python
# server.py
from fastapi import FastAPI
from autogen import ConversableAgent, LLMConfig

app = FastAPI()
llm_config = LLMConfig({"model": "gpt-4o-mini"})

@app.post("/chat")
async def chat(message: str):
    user = ConversableAgent("user", human_input_mode="NEVER")
    agent = ConversableAgent(
        "assistant",
        system_message="You are helpful.",
        llm_config=llm_config,
        human_input_mode="NEVER",
    )
    result = user.run(agent, message=message, max_turns=2)
    result.process()
    return {"response": result.summary}

@app.get("/health")
async def health():
    return {"status": "ok"}
```

The `/health` endpoint is critical — Cloud Run and Kubernetes use it to know if your service is alive. Skip it and your deployments will be flying blind.

### Build and Run

Test locally before you deploy. There's no faster way to burn time than debugging a container in the cloud.

```bash
# Build the image
docker build -t my-agent .

# Run locally (test before deploying)
docker run -p 8080:8080 -e OPENAI_API_KEY=sk-... my-agent

# Test it
curl -X POST "http://localhost:8080/chat?message=Hello"
```

Notice: the API key is passed as an environment variable, **never** baked into the image. This is non-negotiable for production.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# You can test the agent logic locally before containerizing
from autogen import ConversableAgent, LLMConfig

llm_config = LLMConfig({"model": "gpt-4o-mini"})

agent = ConversableAgent(
    "assistant",
    system_message="You are a helpful assistant. Be concise.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

user = ConversableAgent(name="user", human_input_mode="NEVER")

result = user.run(
    agent,
    message="What are the top 3 things to check before deploying an AI agent to production?",
    max_turns=2,
)
result.process()
print(result.summary)

## Part 2 — GCP Agent Starter Pack

You could wire up all the production infrastructure yourself — Terraform, CI/CD, tracing, health checks. Or you could start from a template that's already done it. Google provides the **GCP Agent Starter Pack** with an AG2 template that handles the plumbing so you can focus on your agent logic.

Here's what you get out of the box.

| Component | What it does |
|-----------|-------------|
| **Terraform** | Infrastructure as code — provisions Cloud Run, IAM, networking, secrets |
| **CI/CD Pipelines** | Cloud Build configs for automated testing and deployment |
| **OpenTelemetry** | Distributed tracing and metrics out of the box |
| **A2A Protocol** | Agent-to-Agent communication support (see Part 3) |
| **Dockerfile** | Production-optimized container configuration |
| **Tests** | Integration and unit test scaffolding |

Getting started takes three commands.

```bash
# Clone the starter pack
git clone https://github.com/GoogleCloudPlatform/agent-starter-pack.git

# Navigate to the AG2 template
cd agent-starter-pack/templates/ag2

# Follow the README for GCP project setup and deployment
```

This template handles all the production concerns that would take weeks to build from scratch.

## Part 3 — A2A Protocol (Agent-to-Agent)

Once your agent is deployed as a service, it can talk to other agents — even ones built with different frameworks. The **A2A Protocol** makes this possible over standard HTTP(S).

This matters because it means your AG2 agent can communicate with a LangChain agent, a CrewAI agent, or any A2A-compatible agent. Agents become independent services that discover each other, and you're not locked into a single framework.

The protocol defines three core concepts: **Agent Cards** (metadata describing what an agent can do), **Task lifecycle** (submit, check status, get results), and **Streaming** (real-time updates via SSE).

For full details, see the AG2 blog: `2025-10-27-A2A-Protocol-Support`.

## Production Checklist

Every item on this list exists because someone learned the hard way. Before deploying any agent to production, walk through each one.

- [ ] **Health check endpoint** — `/health` returns 200 OK
- [ ] **Graceful shutdown handling** — finish in-flight requests before stopping
- [ ] **Secrets via environment variables** — never hardcoded, use Secret Manager
- [ ] **Logging and tracing** — structured logs, distributed tracing (Ep 16)
- [ ] **Rate limiting** — protect against abuse and runaway costs
- [ ] **Error handling and retries** — LLM API failures, timeout handling
- [ ] **Cost monitoring** — track token usage and set alerts (Ep 19)
- [ ] **Input validation** — sanitize user inputs before passing to agents
- [ ] **Authentication** — API keys, OAuth, or IAM for your endpoints

Skipping any of these is borrowing against your future self's time.

## Advanced: Kubernetes Deployment

Cloud Run handles most use cases, but if you need fine-grained control — custom networking, persistent connections, multi-service orchestration — Kubernetes is the tool for the job.

```yaml
# k8s-deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: agent-service
spec:
  replicas: 3
  selector:
    matchLabels:
      app: agent-service
  template:
    metadata:
      labels:
        app: agent-service
    spec:
      containers:
      - name: agent
        image: gcr.io/my-project/my-agent:latest
        ports:
        - containerPort: 8080
        livenessProbe:
          httpGet:
            path: /health
            port: 8080
        env:
        - name: OPENAI_API_KEY
          valueFrom:
            secretKeyRef:
              name: agent-secrets
              key: openai-api-key
```

### Blue-Green Deployment

For zero-downtime updates, deploy the new version alongside the old one ("green" alongside "blue"), run smoke tests against the green deployment, switch traffic, and keep blue running as a rollback target. Cloud Run handles this automatically with traffic splitting — one more reason to start there.

## Try It Yourself

Write a `Dockerfile` and `server.py` for your Episode 10 customer service system:

1. Expose a `/chat` endpoint that accepts customer messages
2. Add a `/health` endpoint
3. Pass the API key via environment variable
4. Test locally with `docker run`

**Bonus:** Deploy to Cloud Run using:
```bash
gcloud run deploy agent-service \
  --source . \
  --region us-central1 \
  --set-secrets OPENAI_API_KEY=openai-key:latest
```

## Reference

- Full production template: `gcp-agent-starter-pack/` in the workshop repository
- A2A Protocol blog: [2025-10-27-A2A-Protocol-Support](https://docs.ag2.ai/latest/docs/blog/2025/10/27/A2A-Protocol-Support)
- Cloud Run docs: https://cloud.google.com/run/docs

---

**What's Next:** In Episode 21, you'll explore the redundant pattern — a specialized technique for high-stakes decisions where one answer isn't enough.